# Laya V4.2 — **Replace ALL Full Attention First, THEN All Sliding Attention** with Progressive CeNN Distillation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V42_Progressive_AllFull_Then_AllSliding_CeNN_Colab.ipynb)

### Exact replacement order
**Phase A — Full attention, one-by-one:** **0 → 3 → 6 → 9 → 12 → 15 → 18 → 21 → 24 → 27**. Train a long-range CeNN replacement for each *attention module only*, validate it, save a checkpoint, then train the next layer on activations produced by the current student prefix.

**Phase B — Sliding attention, one-by-one:** **1 → 2 → 4 → 5 → 7 → 8 → 10 → 11 → 13 → 14 → 16 → 17 → 19 → 20 → 22 → 23 → 25 → 26**. Train local CeNN to replace only the attention module using the progressively changed full-attention and sliding-attention prefix.

### Preserve original residual topology, no FFN compression
For EVERY encoder layer, including full and sliding:
\`attn_input = LN_att(hidden)\` → \`attn_out = CeNN(attn_input)\` → \`attn_residual = hidden + attn_out\` → \`FFN(LN_mlp(attn_residual))\` → \`output = attn_residual + FFN_output\`. **ModernBERT's original block performs both residual additions. This notebook replaces \`layer.attn\`, NOT \`layer\`, so both pretrained residuals, normalization modules, pretrained MLP and classifier remain exactly in place.** Verify by forward hooks before training.

The CeNN approximates the original **value mixing and output projection**. It reuses the frozen pretrained attention's V projection and Wo projection, trains lightweight depthwise local and long-range convolutions and an optional gated global summary. Full layers use RF1025 long-range CeNN; sliding layers use a local radius-64 and radius-128 two-scale CeNN. No explicit per-query key routing or duplicate pretrained model.

### Source-derived motivation
V4.1, group 18–20: **95.5%** held-out teacher agreement, **83.5%** gold vs **86.5%** teacher, about **2.12×** speedup for the compressed sliding pair but only **1.02–1.04×** whole-model speed. Training the FFN separately helped substantially; therefore V4.2 **does not compress FFNs at all** during attention research.

### Safe experiment design
- **TRAIN/PROBE/AUDIT are disjoint by case** from source TRAIN. Official TEST untouched.
- At each layer, capture target outputs of the **native attention on inputs from the current compressed prefix**. The frozen native QKV/rotary/SDPA teacher supplies the local target. Train only that layer's small CeNN.
- For the explicit goal of **replacing all 28 attention modules**, default \`SELECTION_MODE="research_all"\` keeps every trained replacement, even when the intermediate quality gate fails. **This is experimental, not deployment-safe**. Switch to \`"quality_gated"\` to reject a trained layer if it lowers whole-model decision quality or does not improve latency.
- After each layer: record local cosine/NMSE, full-model PROBE teacher agreement/gold accuracy/JS, and matched GPU timing; checkpoint the adapter and progress. Resume already-completed layers after a cell rerun or disconnect. Optional Drive-backed storage survives VM deletion.
- Test all full layers as one collective model **before** replacing any sliding layer. Then continue sliding sequentially. At the very end, evaluate the final architecture on the **untouched AUDIT once** with paired CUDA timing, comparison to the original native model, final quality gates and adapter-only checkpoint.
- **No guarantee** of accuracy recovery or speedup from CeNN. A radius-512 receptive field does not reproduce arbitrary full attention. Audit is never used to select configurations.

This notebook runs 28 training stages, so a full run on T4 may take hours. First run with \`STEPS_FULL=80\` and \`STEPS_SLIDING=60\` for architecture screening, then increase steps on an unchanged configuration after inspecting the result.


In [ ]:
#@title 1. Install pinned V4.2 dependencies and GPU check
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Frozen Laya checkpoint + progressive full-then-sliding settings
from pathlib import Path
from contextlib import contextmanager
import math,copy,json,random,hashlib,time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES,build_checkpoint_model,build_sequence,collate_items,
    normalize_question,full_attention_indices,
)
from tinycenn_lm.decision_training import require_finite

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2042
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500 #@param {type:"integer"}
DEV_CASES=240 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
RANK=96 #@param {type:"integer"}
STEPS_FULL=100 #@param {type:"integer"}
STEPS_SLIDING=80 #@param {type:"integer"}
LOCAL_TRAIN_BATCHES=16 #@param {type:"integer"}
LOCAL_PROBE_BATCHES=4 #@param {type:"integer"}
PROBE_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
LEARNING_RATE=0.0003 #@param {type:"number"}
LOG_EVERY=25 #@param {type:"integer"}
BENCH_REPEATS=3 #@param {type:"integer"}
SELECTION_MODE="research_all" #@param ["research_all", "quality_gated"]
MIN_AGREEMENT=.95 #@param {type:"number"}
MAX_GOLD_LOSS_PP=1.0 #@param {type:"number"}
REQUIRE_SPEEDUP_IN_GATED_MODE=False #@param {type:"boolean"}
RESUME=True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}

OUT=Path("/content/Laya_V42_AllFull_Then_Sliding")
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V42")
OUT.mkdir(parents=True,exist_ok=True)
assert 1<=BATCH<=4 and 16<=RANK<=256
assert 10<=STEPS_FULL and 10<=STEPS_SLIDING and LOCAL_TRAIN_BATCHES>=2
assert LOG_EVERY>=1 and BENCH_REPEATS>=2 and TRAIN_CASES+DEV_CASES<=1200
assert SELECTION_MODE in ("research_all","quality_gated")
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"
    ]
))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
ORIGINAL_LAYERS=tuple(teacher.encoder.layers)
ORIGINAL_ATTN=tuple(layer.attn for layer in ORIGINAL_LAYERS)
FULL=tuple(full_attention_indices(teacher))
SLIDING=tuple(i for i in range(28) if i not in FULL)
assert FULL==(0,3,6,9,12,15,18,21,24,27)
assert len(SLIDING)==18
assert int(teacher.encoder.config.local_attention)//2==64
assert MAX_LEN<=int(source_cfg.get("max_len",MAX_LEN))
ORDER=FULL+SLIDING
print("PHASE A full:",FULL)
print("PHASE B sliding:",SLIDING)
print("One original Laya model frozen. T4 GPU",torch.cuda.get_device_name(0))
print("Selection mode:",SELECTION_MODE,"experimental forced-all" if SELECTION_MODE=="research_all" else "gated")


In [ ]:
#@title 3. Case-disjoint TRAIN / PROBE / AUDIT data + batch utilities
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V4.2 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items)>=max(PROBE_DECISIONS,LOCAL_PROBE_BATCHES*BATCH),"Insufficient probe decisions"
assert len(audit_items)>=AUDIT_DECISIONS,"Insufficient audit decisions"
print("official test remains untouched")


In [ ]:
#@title 4. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 5. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 6. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 7. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 8. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 9. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 10. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 11. Implementation
# Additional code follows in the next commit.
